# 04: VisA üzerinde PatchCore, simüle ışık testi, ONNX ve Gradio demosu

VisA veri setinde 4 kategoride PatchCore (anomalib 2.7.0), kaju (cashew) üzerinde parlaklık değişimi testi,
ONNX Runtime ile CPU hız ölçümü ve Gradio demosu.

Veri seti: VisA (Zou et al., ECCV 2022). Lisans bilgisi kaynaklara göre CC BY 4.0 / CC BY-NC-SA 4.0 olarak geçiyor.

In [1]:
!pip install anomalib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 583.9 kB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 709.3 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.9/803.9 kB 740.1 kB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 2.8 MB/s  0:00:10m0:00:0100:01m
  Created wheel for freia: filename=freia-0.2-py3-none-any.whl size=42849 sha256=f7728daeef3e1ce22de77e743b14233e54a700b6e1d576578887a8d3ddcffdfc
  Stored in directory: /root/.cache/pip/wheels/b1/66/ba/29fd62b92bba7c42af99ae475baafa7d205b2c3e7fcc8670b7
Successfully built freia
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [anomalib]6/7 [anomalib]]s]]


## 1. PatchCore, VisA (4 kategori)
anomalib'in `Visa` sınıfı veriyi kendisi indirdi. Notebook içinde ilerleme çubuğu hatası çıktığı için
kod betik dosyası olarak çalıştırıldı.

| Kategori | Image AUROC |
|---|---|
| cashew | 0.966 |
| capsules | 0.735 |
| pcb1 | 0.931 |
| macaroni1 | 0.807 |
| **Ortalama** | **0.859** |

Küçük kusurlu ve çok nesneli sahnelerde (capsules, macaroni1) performans düştü.
Nedeni doğrulanmadı (olası: küçük kusurlar ve varsayılan çözünürlük). Ayar denenmedi.


In [2]:
%%writefile visa_calistir.py
import sys, json, os
import torch
from sklearn.metrics import roc_auc_score
from anomalib.data import Visa
from anomalib.models import Patchcore
from anomalib.engine import Engine

DOSYA = "/kaggle/working/visa_sonuclar.json"

def calistir(kategori):
    dm = Visa(root="/kaggle/working/datasets/visa", category=kategori,
              train_batch_size=16, eval_batch_size=16, num_workers=2)
    model = Patchcore()
    engine = Engine(default_root_dir=f"/kaggle/working/results/visa_{kategori}")
    engine.fit(datamodule=dm, model=model)
    tahm = engine.predict(model=model, dataloaders=dm.test_dataloader())
    skor   = torch.cat([p.pred_score.flatten().cpu() for p in tahm]).numpy()
    etiket = torch.cat([p.gt_label.flatten().cpu() for p in tahm]).numpy()
    return float(roc_auc_score(etiket, skor))

for k in sys.argv[1:]:
    a = calistir(k)
    print("SONUC", k, "image AUROC:", round(a, 4), flush=True)
    s = json.load(open(DOSYA)) if os.path.exists(DOSYA) else {}
    s[k] = a
    json.dump(s, open(DOSYA, "w"))

Writing visa_calistir.py


In [3]:
!python visa_calistir.py cashew 2>&1 | grep -E "SONUC|Error|Traceback"

^C


In [ ]:
!python visa_calistir.py capsules pcb1 macaroni1 2>&1 | grep -E "SONUC|Error|Traceback"

^C


## 2. Simüle ışık testi (cashew)
Model normal ışıkta eğitildi, eşik normal ışıkta sağlam skorlarının 95. yüzdeliğinden seçildi ve sabit tutuldu.
Test görüntülerinin parlaklığı çarpanla değiştirildi (düzgün parlaklık simülasyonu, gerçek ışık değişimi değil).
Test: 50 sağlam, 100 kusurlu.

| Parlaklık | AUROC | Yanlış alarm | Yakalanan kusur |
|---|---|---|---|
| x0.5 | 0.965 | 0.36 | 1.00 |
| x0.75 | 0.967 | 0.06 | 0.85 |
| x1.0 | 0.961 | 0.06 | 0.78 |
| x1.25 | 0.969 | 0.28 | 0.98 |
| x1.5 | 0.956 | 1.00 | 1.00 |

Sıralama kalitesi (AUROC) korundu, ancak sabit eşikle yanlış alarm oranı yükseldi.
Tek kategori, tek çalıştırma, küçük test seti.

In [5]:
%%writefile isik_testi.py
import json, glob
from pathlib import Path
import cv2, numpy as np, torch
from sklearn.metrics import roc_auc_score
from anomalib.data import Visa
from anomalib.models import Patchcore
from anomalib.engine import Engine

KAT = "cashew"
KOK = Path("/kaggle/working/datasets/visa/visa_pytorch")/KAT
CARPANLAR = [0.5, 0.75, 1.0, 1.25, 1.5]

dm = Visa(root="/kaggle/working/datasets/visa", category=KAT,
          train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir="/kaggle/working/results/isik")
engine.fit(datamodule=dm, model=model)

def dosyalar(klasor):
    return sorted(glob.glob(str(klasor/"*.JPG")) + glob.glob(str(klasor/"*.jpg")) + glob.glob(str(klasor/"*.png")))

iyi = dosyalar(KOK/"test/good")
kotu = dosyalar(KOK/"test/bad")
print("TEST sağlam:", len(iyi), "kusurlu:", len(kotu), flush=True)

def skorla(dosya_listesi, carpan, ad):
    hedef = Path(f"/kaggle/working/isik/{ad}_{carpan}")
    hedef.mkdir(parents=True, exist_ok=True)
    for i, f in enumerate(dosya_listesi):
        im = cv2.imread(f).astype(np.float32) * carpan
        cv2.imwrite(str(hedef/f"{i:04d}.png"), np.clip(im, 0, 255).astype(np.uint8))
    tahm = engine.predict(model=model, data_path=str(hedef))
    return torch.cat([p.pred_score.flatten().cpu() for p in tahm]).numpy()

sonuc = {}
esik = None
for c in CARPANLAR:
    si = skorla(iyi, c, "good")
    sk = skorla(kotu, c, "bad")
    if c == 1.0:
        esik = float(np.percentile(si, 95))   # normal ışıkta sağlamların %95'i altında
    sonuc[c] = (si, sk)

for c in CARPANLAR:
    si, sk = sonuc[c]
    auc = roc_auc_score([0]*len(si) + [1]*len(sk), np.concatenate([si, sk]))
    yanlis = float((si > esik).mean())
    yakalama = float((sk > esik).mean())
    print("SONUC", f"parlaklık x{c}", "AUROC:", round(auc, 3),
          "yanlış alarm:", round(yanlis, 3), "yakalanan kusur:", round(yakalama, 3), flush=True)

Writing isik_testi.py


In [6]:
!python isik_testi.py 2>&1 | grep -E "TEST|SONUC|Error|Traceback"

^C


KeyboardInterrupt: 

## 3. ONNX ve CPU hızı
Model ONNX'e dışa aktarıldı (model.onnx + model.onnx.data, 365 MB). Dönüştürmede `Resize` uyarısı çıktı,
ancak dosya çalıştı ve 5 örnek çiftte kusurlu görüntülere sağlamlardan yüksek skor verdi.
Dosyadaki hazır `pred_label` karar eşiği kullanılamadı (hepsine "kusurlu" dedi).

| Ölçüm (Kaggle CPU) | Görüntü başına | FPS |
|---|---|---|
| PyTorch (224x224, batch 1) | ~1055 ms | 0.95 |
| ONNX Runtime (224x224, batch 2) | ~829 ms | 1.21 |

Yaklaşık karşılaştırma, tek ölçüm. Batch ve ön/son işleme koşulları birebir aynı değil.

In [7]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 1.7 MB/s  0:00:14m0:00:0100:01


In [8]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 7.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]2 [onnxscript]


In [9]:
%%writefile onnx_hiz.py
import glob, time, resource
import numpy as np, torch
from anomalib.data import Visa
from anomalib.models import Patchcore
from anomalib.engine import Engine
from anomalib.deploy import ExportType
import onnxruntime as ort

dm = Visa(root="/kaggle/working/datasets/visa", category="cashew",
          train_batch_size=16, eval_batch_size=16, num_workers=2)
model = Patchcore()
engine = Engine(default_root_dir="/kaggle/working/results/onnx")
engine.fit(datamodule=dm, model=model)

try:
    engine.export(model=model, export_type=ExportType.ONNX,
                  export_root="/kaggle/working/export")
except Exception as e:
    print("SONUC export uyarısı:", str(e)[:200], flush=True)
yol = glob.glob("/kaggle/working/export/**/*.onnx", recursive=True)
print("SONUC onnx dosyası:", yol, flush=True)

def olc(f, n=30, isinma=5):
    for _ in range(isinma):
        f()
    t = []
    for _ in range(n):
        b = time.perf_counter(); f(); t.append(time.perf_counter() - b)
    return float(np.median(t))

# PyTorch, CPU (giriş: 224x224, batch 1)
m = model.model.cpu().eval()
xt = torch.rand(1, 3, 224, 224)
def torch_calistir():
    with torch.no_grad():
        m(xt)
t_torch = olc(torch_calistir)
print("SONUC PyTorch CPU:", round(1/t_torch, 2), "FPS,", round(t_torch*1000, 1), "ms (224x224, batch 1)", flush=True)

# ONNX Runtime, CPU
try:
    sess = ort.InferenceSession(yol[0], providers=["CPUExecutionProvider"])
    inp = sess.get_inputs()[0]
    print("SONUC onnx ham giriş şekli:", inp.shape, flush=True)
    sekil = [d if isinstance(d, int) else (1 if i == 0 else 256) for i, d in enumerate(inp.shape)]
    x = np.random.rand(*sekil).astype(np.float32)
    t_onnx = olc(lambda: sess.run(None, {inp.name: x}))
    print("SONUC ONNX CPU:", round(sekil[0]/t_onnx, 2), "FPS,", round(t_onnx*1000, 1),
          "ms, giriş", sekil, flush=True)
except Exception as e:
    print("SONUC ONNX hata:", str(e)[:300], flush=True)

print("SONUC en yüksek bellek (MB):", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024), flush=True)

Writing onnx_hiz.py


In [ ]:
!python onnx_hiz.py 2>&1 | grep -E "SONUC|Error|Traceback"

In [10]:
%%writefile onnx_kontrol.py
import glob, time
import numpy as np, cv2
import onnxruntime as ort

yol = glob.glob("/kaggle/working/export/**/*.onnx", recursive=True)[0]
sess = ort.InferenceSession(yol, providers=["CPUExecutionProvider"])
ad = sess.get_inputs()[0].name
print("SONUC çıkış adları:", [o.name for o in sess.get_outputs()], flush=True)

def olc(x, n=20, isinma=3):
    for _ in range(isinma): sess.run(None, {ad: x})
    t = []
    for _ in range(n):
        b = time.perf_counter(); sess.run(None, {ad: x}); t.append(time.perf_counter() - b)
    return float(np.median(t))

for boyut in [224, 256]:
    x = np.random.rand(2, 3, boyut, boyut).astype(np.float32)
    s = olc(x)
    print(f"SONUC ONNX {boyut}x{boyut}, batch 2: {s*1000:.0f} ms, görüntü başına {s*500:.0f} ms, {2/s:.2f} FPS", flush=True)

KOK = "/kaggle/working/datasets/visa/visa_pytorch/cashew/test"
iyi  = sorted(glob.glob(KOK + "/good/*"))[:5]
kotu = sorted(glob.glob(KOK + "/bad/*"))[:5]

def oku(f):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (256, 256)).astype(np.float32) / 255.0
    return im.transpose(2, 0, 1)

for g, b in zip(iyi, kotu):
    x = np.stack([oku(g), oku(b)])
    cikti = sess.run(None, {ad: x})
    satir = []
    for o, c in zip(sess.get_outputs(), cikti):
        satir.append(f"{o.name}{list(c.shape)} sağlam={float(c[0].max()):.3f} kusurlu={float(c[1].max()):.3f}")
    print("SONUC", " | ".join(satir), flush=True)

Writing onnx_kontrol.py


In [ ]:
!python onnx_kontrol.py 2>&1 | grep -E "SONUC|Traceback|Error"

In [11]:
!ls -lh /kaggle/working/export/weights/onnx/

ls: cannot access '/kaggle/working/export/weights/onnx/': No such file or directory


In [12]:
!find /kaggle/input -name "model.onnx*"

/kaggle/input/datasets/betlsari0/cashew-onnx/model.onnx
/kaggle/input/datasets/betlsari0/cashew-onnx/model.onnx (1).data


## 4. Eşik ve Gradio demosu
Eşik, 24 sağlam test görüntüsünün 95. yüzdeliğinden seçildi (0.612). Küçük örneklem, üretim ayarı değil.
Demo: görüntü yükle, ısı haritası ve OK/NOK kararı gör, eşiği kaydırıcıyla ayarla.
Isı haritası her görüntü için kendi aralığında ölçeklendiğinden karar skordan verilir, haritadan değil.

In [13]:
!pip install anomalib onnxruntime gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 16.7 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pillow
    Found existing installation: pillow 13.0.0
    Uninstalling pillow-13.0.0:
      Successfully uninstalled pillow-13.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [14]:
import os, shutil
os.makedirs("/kaggle/working/model", exist_ok=True)
G = "/kaggle/input/datasets/betlsari0/cashew-onnx"
shutil.copy(f"{G}/model.onnx", "/kaggle/working/model/model.onnx")
shutil.copy(f"{G}/model.onnx (1).data", "/kaggle/working/model/model.onnx.data")
!ls -lh /kaggle/working/model

total 366M
-rw-rw-rw- 1 root root 336K Oct 26 09:25 model.onnx
-rw-rw-rw- 1 root root 365M Oct 26 09:25 model.onnx.data


In [15]:
from anomalib.data import Visa
dm = Visa(root="/kaggle/working/datasets/visa", category="cashew")
dm.prepare_data()

VisA: 1.93GB [02:32, 12.7MB/s]                               


In [16]:
!ls /kaggle/working/datasets/visa/visa_pytorch/cashew

ground_truth  test  train


In [17]:
import glob, numpy as np, cv2, onnxruntime as ort

MODEL_YOLU = "/kaggle/working/model/model.onnx"
sess = ort.InferenceSession(MODEL_YOLU, providers=["CPUExecutionProvider"])
ad = sess.get_inputs()[0].name

def oku(f):
    im = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    return cv2.resize(im, (256, 256)).astype(np.float32).transpose(2, 0, 1) / 255.0

def skor_iki(a, b):
    return sess.run(None, {ad: np.stack([a, b])})

KOK = "/kaggle/working/datasets/visa/visa_pytorch/cashew"
iyi = sorted(glob.glob(KOK + "/train/good/*"))[:30]
sk = []
for i in range(0, 30, 2):
    sk += list(skor_iki(oku(iyi[i]), oku(iyi[i+1]))[0])
ESIK = float(np.percentile(sk, 95))
print("sağlam skor aralığı:", round(min(sk), 3), round(max(sk), 3), "| varsayılan eşik:", round(ESIK, 3))

sağlam skor aralığı: 0.485 0.628 | varsayılan eşik: 0.6


In [19]:
import glob, shutil, os
os.makedirs("/kaggle/working/ornek", exist_ok=True)
K = "/kaggle/working/datasets/visa/visa_pytorch/cashew/test"
for tur in ["good", "bad"]:
    for i, f in enumerate(sorted(glob.glob(f"{K}/{tur}/*"))[:3]):
        shutil.copy(f, f"/kaggle/working/ornek/{tur}_{i}{os.path.splitext(f)[1]}")
!ls -lh /kaggle/working/ornek

total 1.9M
-rw-r--r-- 1 root root 348K Oct 26 09:34 bad_0.JPG
-rw-r--r-- 1 root root 335K Oct 26 09:34 bad_1.JPG
-rw-r--r-- 1 root root 347K Oct 26 09:34 bad_2.JPG
-rw-r--r-- 1 root root 289K Oct 26 09:34 good_0.JPG
-rw-r--r-- 1 root root 302K Oct 26 09:34 good_1.JPG
-rw-r--r-- 1 root root 293K Oct 26 09:34 good_2.JPG


In [25]:
iyi_test = sorted(glob.glob(KOK + "/test/good/*"))
esik_seti = iyi_test[:24]
sk = []
for i in range(0, 24, 2):
    sk += list(skor_iki(oku(esik_seti[i]), oku(esik_seti[i+1]))[0])
ESIK = float(np.percentile(sk, 95))
print("sağlam skor aralığı:", round(min(sk), 3), round(max(sk), 3), "| yeni eşik:", round(ESIK, 3))

sağlam skor aralığı: 0.518 0.651 | yeni eşik: 0.612


In [30]:
arayuz.close()
arayuz = gr.Interface(
    fn=tahmin,
    inputs=[gr.Image(type="numpy", label="Görüntü"),
            gr.Slider(0.4, 0.9, value=0.612, step=0.005, label="Eşik")],
    outputs=[gr.Image(label="Isı haritası"), gr.Textbox(label="Karar")],
    title="Endüstriyel kusur tespiti (PatchCore, VisA cashew)",
)
arayuz.launch(share=True)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1e9f55ff83b2aaaadf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
